In [1]:
import numpy as np
import pandas as pd
from tensorflow import keras
import seaborn as sns
import matplotlib.pyplot as plt
from keras import layers
from sklearn.preprocessing import StandardScaler
import numpy as np
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

In [44]:
#import the datasets to test
def load_data():
    data = np.load("roads_canids_windows_200hz_3s.npz")
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_test"]
    y_test  = data["y_test"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = load_data()

KeyboardInterrupt: 

In [39]:
def standardize(X_train, y_train, X_test, y_test):
    # Step 1: Clip outliers (important for ROAD)
    X_train = np.clip(X_train, -1e6, 1e6)
    X_test  = np.clip(X_test,  -1e6, 1e6)
    # Step 2: Standardize features
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, X_train.shape[-1])
    X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled  = scaler.transform(X_test_flat)
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test  = X_test_scaled.reshape(X_test.shape)
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = standardize(X_train, y_train, X_test, y_test)

In [46]:
def create_model(): #this is the same model we'll always use for all. 
    model = keras.Sequential()
    model.add(layers.Input(shape=(600, 23)))
    model.add(layers.Conv1D(16, 4, activation='relu'))
    model.add(layers.GlobalMaxPooling1D())
    model.add(layers.Dense(1, activation='sigmoid')) #output 1 bc we only have 2 labels: attack or not attack
    return model
model = create_model()

In [47]:
#train data 
b_size = 32
callbacks = [
    ModelCheckpoint("saved_models/best_model_road_cnn.keras", monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-9, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=16, verbose=1, restore_best_weights=True)
]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss = 'binary_crossentropy', 
              metrics = ['accuracy', keras.metrics.AUC(name='auc'), 
              keras.metrics.Precision(name='precision'),  keras.metrics.Recall(name='recall')])
model.fit(X_train, y_train, batch_size = b_size, epochs = 30, validation_split=0.1, callbacks = callbacks, verbose = 1)

Epoch 1/30
315/318 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5510 - auc: 0.6357 - loss: 2.7790 - precision: 0.0730 - recall: 0.5322
Epoch 1: val_accuracy improved from None to 0.57713, saving model to saved_models/best_model_road_cnn.keras
318/318 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.8021 - auc: 0.7186 - loss: 0.9622 - precision: 0.0936 - recall: 0.2865 - val_accuracy: 0.5771 - val_auc: 0.6254 - val_loss: 0.8472 - val_precision: 0.5489 - val_recall: 0.1490 - learning_rate: 0.0010
Epoch 2/30
313/318 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9442 - auc: 0.9382 - loss: 0.1364 - precision: 0.6161 - recall: 0.2869
Epoch 2: val_accuracy improved from 0.57713 to 0.63741, saving model to saved_models/best_model_road_cnn.keras
318/318 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9509 - auc: 0.9489 - loss: 0.1213 - precision: 0.6393 - recall: 0.3108 - val_accuracy: 0.6374 - val_auc: 0.7358 - val_loss: 0.7237 - val_precision: 0.8140 - val_recall: 0.2143 - learning_rate: 

In [49]:
results = model.evaluate(X_test, y_test)
print(dict(zip(model.metrics_names, results)))

436/436 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9854 - auc: 0.9362 - loss: 0.0553 - precision: 0.7500 - recall: 0.4800          
{'loss': 0.05525096878409386, 'compile_metrics': 0.9853742718696594}
